# The Median Line — a straight robust trend vs traditional dip signals
**Jake's "median price" metric, evolved.** Instead of a *rolling* median (which lags like an SMA), fit **one
straight line through ~10 years** of each stock's log-price using **median (least-absolute-deviations, LAD)
regression** — the line that threads the *densest crossover path* because it minimizes absolute distance, so
spikes don't drag it around. Then ask: **is "far below the median line" a better dip-buy signal than the
200-day SMA or "% off the 52-week high"?**

### What it does
For every **current** S&P 500 name with ~10y history, walk-forward (no lookahead) it builds four dip signals
and measures **forward returns** after each fires, at **matched firing frequency** (apples-to-apples):
1. **MEDLINE** — price below its LAD/median trend line by a robust z-score *(the new idea)*
2. **SMA200** — price below the 200-day simple moving average *(tradition)*
3. **PCT** — price ≥ X% below the trailing 52-week high *(tradition)*
4. **ROLLMED** — price below the 200-day *rolling median* *(Jake's existing median-line metric — the incumbent)*

### ⚠️ Read these caveats before trusting anything
- **Survivorship bias, by construction.** We take *today's* index members and follow them back. These are the
  winners — names that dipped and recovered. So this answers *"for quality names that stayed in the index, was
  the median line a better dip trigger?"* — useful for a watchlist, **NOT** a claim about all stocks.
- **No lookahead.** The line is refit only on *trailing* data at each date. A line fit on the full 10y would
  know the future and every dip would look buyable — worthless. This uses trailing windows only.
- **Matched frequency.** A rarer/deeper signal isn't automatically "better." Each signal is thresholded to fire
  on the *same %* of days, so we compare like for like.
- Thin-sample and knob-turning warnings are built in. First backtest = first honest read.


In [ ]:
# 1 · Setup
!pip -q install yfinance >/dev/null 2>&1
import numpy as np, pandas as pd, warnings, sys
warnings.filterwarnings("ignore")
import yfinance as yf
try:
    import statsmodels.api as sm            # for QuantReg (median/LAD regression)
    HAVE_SM = True
except Exception:
    HAVE_SM = False
try:
    from scipy.stats import theilslopes     # robust fallback
    from scipy.optimize import linprog      # LAD via LP fallback
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False
print("statsmodels:", HAVE_SM, "| scipy:", HAVE_SCIPY)


In [ ]:
# 2 · CONFIG  (edit these — single-value changes are iPhone-friendly)
YEARS          = 10          # history to pull / fit the median line over
LINE_METHOD    = "lad"       # "lad" (median regression, THE metric) | "theil" (fast robust) | "ols"
TRAIL_YEARS    = 5           # trailing window the line is refit on (no lookahead)
REFIT_DAYS     = 21          # refit the line every N trading days (21≈monthly). ↑ = faster, coarser
RESAMPLE_LINE  = "W"         # resample to Weekly for the line fit (speed; "" = daily)

Z_MEDLINE      = 1.0         # MEDLINE abs-threshold: dip when price is this many robust-σ below the line
SMA_WIN        = 200         # SMA / rolling-median window (days)
PCT_OFF_HIGH   = 0.10        # PCT dip: 10% below the trailing 252d high
HIGH_WIN       = 252

HORIZONS       = [21, 63, 126, 252]   # forward-return horizons (trading days ≈ 1,3,6,12 months)
MATCH_PCT      = 0.15        # matched-frequency: each signal fires on the ~15% most-dipped name-days
COMMON_SUPPORT = True        # only race on name-days where ALL four signals are defined (fair apples-to-apples)
MIN_HISTORY_D  = 252*8       # require ≥8y of data to include a name

LIMIT_NAMES    = 0           # 0 = whole S&P; set e.g. 60 for a fast test run first
EXAMPLE_TICKER = "AAPL"      # the name to plot at the end
print("CONFIG loaded. LINE_METHOD =", LINE_METHOD, "| whole S&P" if not LIMIT_NAMES else f"| first {LIMIT_NAMES}")


In [ ]:
# 3 · The median line + helpers
def fit_line(x, y):
    '''Fit y ≈ a + b·x. Returns (a,b). Median/LAD regression = the 'densest crossover path' line.'''
    x = np.asarray(x, float); y = np.asarray(y, float)
    if len(x) < 5:
        return np.nan, np.nan
    if LINE_METHOD == "lad" and HAVE_SM:
        try:
            X = sm.add_constant(x)
            r = sm.QuantReg(y, X).fit(q=0.5)          # q=0.5 == median regression == LAD
            return float(r.params[0]), float(r.params[1])
        except Exception:
            pass
    if LINE_METHOD == "lad" and HAVE_SCIPY:            # LP fallback: min Σ|resid|
        try:
            n = len(x)
            # vars: a, b, u_1..u_n ;  min Σu ; u ≥ ±(y-(a+bx))
            c = np.r_[0, 0, np.ones(n)]
            A = np.zeros((2*n, 2+n)); bnd = np.zeros(2*n)
            for i in range(n):
                A[i,0]=-1; A[i,1]=-x[i]; A[i,2+i]=-1; bnd[i]=-y[i]      # (a+bx) - u ≤ y
                A[n+i,0]=1; A[n+i,1]=x[i]; A[n+i,2+i]=-1; bnd[n+i]=y[i] #-(a+bx) - u ≤ -y
            res = linprog(c, A_ub=A, b_ub=bnd, bounds=[(None,None)]*2+[(0,None)]*n, method="highs")
            if res.success:
                return float(res.x[0]), float(res.x[1])
        except Exception:
            pass
    if LINE_METHOD in ("lad","theil") and HAVE_SCIPY:  # Theil-Sen (robust median of slopes)
        try:
            b, a, *_ = theilslopes(y, x)
            return float(a), float(b)
        except Exception:
            pass
    b, a = np.polyfit(x, y, 1)                          # OLS last resort
    return float(a), float(b)

def robust_scale(r):
    r = r[np.isfinite(r)]
    if len(r) < 5: return np.nan
    return 1.4826 * np.median(np.abs(r - np.median(r))) or np.nan

def medline_full(close):
    '''Walk-forward fit of log-price vs its trailing median (LAD) line. No lookahead.
       Returns (z, line_log, scale): z negative = price below the median line.'''
    lp = np.log(close.astype(float))
    idx = lp.index
    line = pd.Series(np.nan, index=idx); scale = pd.Series(np.nan, index=idx)
    yr = (idx - idx[0]).days / 365.25
    yrs = pd.Series(yr, index=idx)
    start_i = np.searchsorted(idx, idx[0] + pd.Timedelta(days=int(TRAIL_YEARS*365.25)))
    refit_pts = list(range(start_i, len(idx), REFIT_DAYS))
    if refit_pts and refit_pts[-1] != len(idx)-1:
        refit_pts.append(len(idx)-1)
    for k, ri in enumerate(refit_pts):
        d0 = idx[ri] - pd.Timedelta(days=int(TRAIL_YEARS*365.25))
        win = lp[(idx >= d0) & (idx <= idx[ri])]
        if len(win) < 60: continue
        w = win.resample(RESAMPLE_LINE).last().dropna() if RESAMPLE_LINE else win
        wx = (w.index - idx[0]).days / 365.25
        a, b = fit_line(wx, w.values)
        if not np.isfinite(a): continue
        s = robust_scale(w.values - (a + b*wx))
        lo = ri
        hi = refit_pts[k+1] if k+1 < len(refit_pts) else len(idx)
        seg = idx[lo:hi]
        line.loc[seg] = a + b*yrs.loc[seg].values
        scale.loc[seg] = s
    return (lp - line) / scale, line, scale      # z_t : negative = below the median line

def medline_z(close):
    return medline_full(close)[0]


In [ ]:
# 4 · Universe — current S&P 500, followed back
def sp500_tickers():
    try:
        t = pd.read_html("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies")[0]
        syms = [s.replace(".", "-") for s in t["Symbol"].astype(str)]
        print(f"Wikipedia: {len(syms)} current constituents")
        return syms
    except Exception as e:
        print("Wikipedia failed, using fallback core list:", str(e)[:60])
        return ("AAPL MSFT NVDA AMZN GOOGL META AVGO TSLA LLY JPM V UNH XOM MA COST HD PG JNJ "
                "ABBV WMT MRK NFLX ADBE CRM AMD KO PEP TMO ORCL").split()
TICKERS = sp500_tickers()
if LIMIT_NAMES: TICKERS = TICKERS[:LIMIT_NAMES]
print("using", len(TICKERS), "names")


In [ ]:
# 5 · Download ~10y daily closes (batched, token-free)
px = yf.download(TICKERS, period=f"{YEARS}y", auto_adjust=True, progress=True)["Close"]
if isinstance(px, pd.Series): px = px.to_frame()
px = px.dropna(how="all")
keep = [c for c in px.columns if px[c].dropna().shape[0] >= MIN_HISTORY_D]
px = px[keep]
print(f"{len(keep)} names with ≥{MIN_HISTORY_D} days ({MIN_HISTORY_D/252:.0f}y). Dropped {len(TICKERS)-len(keep)}.")


In [ ]:
# 6 · Build the pooled signal + forward-return panel (walk-forward)
def build_panel(close):
    close = close.dropna()
    z   = medline_z(close)                                    # MEDLINE depth
    sma = close.rolling(SMA_WIN).mean()
    rmd = close.rolling(SMA_WIN).median()                     # Jake's existing rolling median
    hi  = close.rolling(HIGH_WIN).max()
    df = pd.DataFrame({"close": close})
    df["medline_depth"] = -z                                  # +ve = below line (deeper dip)
    df["sma_depth"]     = (sma - close) / sma                 # +ve = below SMA
    df["rollmed_depth"] = (rmd - close) / rmd                 # +ve = below rolling median
    df["pct_depth"]     = (hi - close) / hi                   # +ve = below the high (drawdown)
    for H in HORIZONS:
        df[f"fwd_{H}"] = close.shift(-H) / close - 1.0
    return df

panels = {}
for i, t in enumerate(px.columns):
    try:
        panels[t] = build_panel(px[t])
    except Exception as e:
        pass
    if (i+1) % 50 == 0: print(f"  {i+1}/{len(px.columns)} processed")
pool = pd.concat([p.assign(ticker=t) for t, p in panels.items()])
print("pooled name-days:", f"{len(pool):,}", "| names:", len(panels))


In [ ]:
# 7 · MATCHED-FREQUENCY drag race (the headline table)
REPORT = []
def out(s=""):
    print(s); REPORT.append(s)

DEPTHS = {"MEDLINE": "medline_depth", "SMA200": "sma_depth",
          "PCT_off_high": "pct_depth", "ROLLMED": "rollmed_depth"}

# COMMON SUPPORT: race only on name-days where ALL four signals are defined (fair apples-to-apples).
# Without this, MEDLINE (needs a trailing window before it exists) gets judged on a later sub-sample.
if COMMON_SUPPORT:
    mask = pool[list(DEPTHS.values())].notna().all(axis=1)
    race = pool[mask].copy()
    out(f"common-support rows (all 4 signals defined): {len(race):,} of {len(pool):,} "
        f"({len(race)/len(pool):.0%})")
else:
    race = pool
    out("NOTE: COMMON_SUPPORT off — each signal judged on its own defined domain (not strictly fair).")

out("="*74)
out(f"MATCHED-FREQUENCY DIP DRAG RACE  —  each signal fires on the top {MATCH_PCT:.0%} most-dipped name-days")
out(f"LINE_METHOD={LINE_METHOD}  trail={TRAIL_YEARS}y refit/{REFIT_DAYS}d  |  survivorship-conditioned")
out("="*74)
for H in HORIZONS:
    base = race[f"fwd_{H}"].median()
    bpos = (race[f"fwd_{H}"] > 0).mean()
    out(f"\n── forward {H}d (≈{H/21:.0f}mo)   BASELINE all-days: median {base:+.2%}  |  win {bpos:.0%}")
    out(f"   {'signal':<14}{'N':>8}{'med fwd':>10}{'mean':>9}{'win%':>7}{'EDGE vs base':>14}")
    rows = {}
    for name, col in DEPTHS.items():
        d = race[[col, f"fwd_{H}"]].dropna()
        if len(d) < 200:
            rows[name] = None; continue
        thr = d[col].quantile(1 - MATCH_PCT)
        sig = d[d[col] >= thr]
        rows[name] = (len(sig), sig[f"fwd_{H}"].median(), sig[f"fwd_{H}"].mean(),
                      (sig[f"fwd_{H}"] > 0).mean())
    best = max((r[1] for r in rows.values() if r), default=None)
    for name, r in rows.items():
        if r is None:
            out(f"   {name:<14}{'thin':>8}"); continue
        n, med, mean, win = r
        flag = "  <<<" if best is not None and med == best else ""
        out(f"   {name:<14}{n:>8,}{med:>+10.2%}{mean:>+9.2%}{win:>7.0%}{med-base:>+13.2%}{flag}")
out("\nRead: higher EDGE = that dip definition marked better forward entries (at equal firing rate).")
out("'<<<' = best signal for that horizon. If MEDLINE doesn't lead, the straight line isn't beating tradition.")


In [ ]:
# 8 · Per-name win rate (so one mega-cap can't carry the result)
out("\n" + "="*74)
out(f"PER-NAME: in how many names does MEDLINE beat each rival? (median fwd_63, matched {MATCH_PCT:.0%})")
out("="*74)
H = 63
wins = {"SMA200":0, "PCT_off_high":0, "ROLLMED":0}; n_ok = 0
for t, p in panels.items():
    p = p.dropna(subset=[f"fwd_{H}"])
    if len(p) < 300: continue
    def sig_med(col):
        d = p[[col, f"fwd_{H}"]].dropna()
        if len(d) < 60: return np.nan
        thr = d[col].quantile(1-MATCH_PCT)
        return d[d[col] >= thr][f"fwd_{H}"].median()
    m = sig_med("medline_depth")
    if not np.isfinite(m): continue
    n_ok += 1
    for rival, col in [("SMA200","sma_depth"),("PCT_off_high","pct_depth"),("ROLLMED","rollmed_depth")]:
        r = sig_med(col)
        if np.isfinite(r) and m > r: wins[rival] += 1
for rival, w in wins.items():
    out(f"   MEDLINE beats {rival:<13} in {w:>3}/{n_ok} names  ({w/max(n_ok,1):.0%})")
out("   >50% = the median line is the better dip trigger for the median name (not just in aggregate).")


In [ ]:
# 9 · SEE it — one name: price, median line, ±bands, dip markers
import matplotlib.pyplot as plt
t = EXAMPLE_TICKER if EXAMPLE_TICKER in px.columns else px.columns[0]
c = px[t].dropna()
z, line_log, scale = medline_full(c)          # the ACTUAL walk-forward median line
line = np.exp(line_log)
upper = np.exp(line_log + Z_MEDLINE*scale); lower = np.exp(line_log - Z_MEDLINE*scale)
fig, ax = plt.subplots(2, 1, figsize=(12, 8), sharex=True, gridspec_kw={"height_ratios":[2,1]})
ax[0].plot(c.index, c.values, lw=1, color="black", label=f"{t} close")
ax[0].plot(line.index, line.values, lw=1.6, color="tab:blue", label=f"median line ({LINE_METHOD}, walk-fwd)")
ax[0].fill_between(line.index, lower.values, upper.values, color="tab:blue", alpha=.10,
                   label=f"±{Z_MEDLINE}σ band")
ax[0].plot(c.index, c.rolling(SMA_WIN).mean(), lw=.8, alpha=.6, color="tab:orange", label=f"{SMA_WIN}d SMA")
dips = c[z <= -Z_MEDLINE]
ax[0].scatter(dips.index, dips.values, s=12, color="red", zorder=5, label=f"MEDLINE dip (z≤-{Z_MEDLINE})")
ax[0].set_title(f"{t} — the median line vs traditional dip signals"); ax[0].legend(fontsize=8)
ax[0].set_yscale("log")
ax[1].plot(z.index, z.values, lw=.8, color="tab:blue"); ax[1].axhline(0, color="k", lw=.5)
ax[1].axhline(-Z_MEDLINE, color="red", ls="--", lw=.7); ax[1].set_ylabel("z vs median line")
plt.tight_layout(); plt.show()
zl = z.dropna()
print(f"{t}: currently {zl.iloc[-1]:+.2f}σ vs its {LINE_METHOD} median line "
      f"({'BELOW — dip zone' if zl.iloc[-1] <= -Z_MEDLINE else 'BELOW' if zl.iloc[-1] < 0 else 'ABOVE'}).")


In [ ]:
# 10 · Export the report + panel
with open("median_line_report.txt", "w") as f:
    f.write("\n".join(REPORT))
pool.reset_index().rename(columns={"index":"date"}).to_csv("median_line_panel.csv.gz",
                                                          index=False, compression="gzip")
print("saved median_line_report.txt  +  median_line_panel.csv.gz")
print("\nPaste the MATCHED-FREQUENCY table + the per-name win rates back into the vault chat for the read.")


## How to read it
- **The headline is the EDGE column** in the matched-frequency table. It's each dip signal's forward return
  **minus** the all-days baseline, at equal firing frequency. Positive = the dip marked a better-than-average
  entry. **The winner per horizon gets `<<<`.**
- **MEDLINE leads → your straight median line beats the 200-day and %-off-high** as a dip trigger. If SMA or
  ROLLMED leads, the fancy line isn't earning its keep and tradition wins — log that honestly.
- **Per-name win rate >50%** means it works for the *typical* name, not just because one mega-cap dominates
  the pool. Both need to agree before this becomes a real tool.
- **Knobs that matter most:** `LINE_METHOD` ("lad" vs "theil" vs "ols" — does robustness matter?), `TRAIL_YEARS`,
  `MATCH_PCT`. **Do not knob-turn until something goes green** — that's data-mining. Legit robustness checks:
  rerun with `LINE_METHOD="theil"`, a different `MATCH_PCT`, and `LIMIT_NAMES=60` vs whole index.
- **Survivorship** still colors everything: this is "for names that stayed in the index." A clean test would
  need point-in-time membership (a paid dataset) — out of scope, flagged so we never oversell the result.
